# Module 14: Async Human Approval - 02: Timeout and escalation

> **MLCourse - Agentic AI - Agent Patterns**

Notebook 01 built the core pattern: enqueue durably, suspend, resume from a
separate process by `thread_id`. It quietly assumed the reviewer eventually
answers. **They might not.** They are on leave, the notification never
arrived, or the request simply got lost in a busy inbox.

This notebook adds the piece every real approval system needs: **what happens
when nobody answers in time.**

### What you will learn

1. Why a suspended thread with no timeout policy is a liability, not a
   convenience.
2. Detecting an overdue request from the queue's own timestamps.
3. **Escalation**: routing an overdue request to a different, usually more
   senior, reviewer - without losing the original request or double-booking
   the decision.
4. Composing timeout + escalation into a single poll-and-resume loop.

Still no LLM calls, no API key, no web server.

### Setup: rebuild the queue and a fresh graph


In [ ]:
import json
import os
import sqlite3
import time
import uuid
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Optional, TypedDict

from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

QUEUE_PATH = Path("escalation_queue.json")
DB_PATH = "escalation_demo.db"
if QUEUE_PATH.exists():
    QUEUE_PATH.unlink()
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

checkpointer = SqliteSaver(sqlite3.connect(DB_PATH, check_same_thread=False))
print("timeout/escalation demo ready -- no API key needed")


### 1. The queue, extended with an escalation trail

The request record from notebook 01 grows two fields: `assigned_to` (who is
currently on the hook) and `escalation_level`. Nothing about the underlying
mechanism changes - it is the same durable JSON store - but these two fields
are what let a poller decide *who* to notify next, and let you audit exactly
who had the request at each point.

### The queue, with escalation fields


In [ ]:
@dataclass
class ApprovalRequest:
    request_id: str
    thread_id: str
    summary: str
    payload: dict
    created_at: float
    assigned_to: str
    escalation_level: int = 0
    status: str = "pending"
    decided_by: Optional[str] = None
    decided_at: Optional[float] = None
    reason: Optional[str] = None
    escalation_log: list = field(default_factory=list)


class ApprovalQueue:
    def __init__(self, path: Path):
        self.path = path
        if not self.path.exists():
            self._write({})

    def _read(self) -> dict:
        return json.loads(self.path.read_text()) if self.path.exists() else {}

    def _write(self, data: dict) -> None:
        self.path.write_text(json.dumps(data, indent=2))

    def enqueue(self, thread_id, summary, payload, assigned_to) -> str:
        req = ApprovalRequest(request_id=str(uuid.uuid4())[:8], thread_id=thread_id,
                              summary=summary, payload=payload, created_at=time.time(),
                              assigned_to=assigned_to)
        data = self._read()
        data[req.request_id] = asdict(req)
        self._write(data)
        return req.request_id

    def pending(self) -> list:
        return [r for r in self._read().values() if r["status"] == "pending"]

    def get(self, request_id: str) -> dict:
        return self._read()[request_id]

    def decide(self, request_id, approved, reviewer, reason="") -> None:
        data = self._read()
        data[request_id].update(status="approved" if approved else "rejected",
                                decided_by=reviewer, decided_at=time.time(), reason=reason)
        self._write(data)

    def escalate(self, request_id: str, new_assignee: str, note: str) -> None:
        """Reassign an overdue request. The ORIGINAL request_id is preserved --
        this is a reassignment, not a new request -- so anything already
        watching this id (like a resume loop) keeps working unmodified."""
        data = self._read()
        r = data[request_id]
        r["escalation_log"].append({
            "from": r["assigned_to"], "to": new_assignee,
            "note": note, "at": time.time(),
        })
        r["assigned_to"] = new_assignee
        r["escalation_level"] += 1
        self._write(data)


queue = ApprovalQueue(QUEUE_PATH)
print("escalation-aware queue ready")


### 2. Detecting an overdue request

There is no special "timeout" signal from the checkpointer - a suspended
thread just sits there indefinitely, which is correct (it costs nothing to
wait). Timeout has to be **computed** by a poller, by comparing `created_at`
(or the time of the *last* escalation) against a policy-defined SLA.

### SLA policy and overdue detection


In [ ]:
# Real policy would vary by amount, risk, or team. Kept as one constant here
# so the mechanism, not the policy table, stays the focus.
SLA_SECONDS = 3.0     # deliberately tiny so the demo doesn't sit and wait


def is_overdue(req: dict) -> bool:
    """A request is overdue if it has been sitting with its CURRENT assignee
    (not its original creation time) longer than the SLA."""
    last_touch = req["escalation_log"][-1]["at"] if req["escalation_log"] else req["created_at"]
    return req["status"] == "pending" and (time.time() - last_touch) > SLA_SECONDS


thread_id = "purchase-needs-escalation"
cfg = {"configurable": {"thread_id": thread_id}}


class PurchaseState(TypedDict):
    item: str
    amount: float
    request_id: str
    verdict: str


def request_approval(state: PurchaseState) -> dict:
    req_id = queue.enqueue(thread_id=thread_id,
                           summary=f"Approve: {state['item']} (${state['amount']:.2f})",
                           payload={"item": state["item"]}, assigned_to="l1-reviewer@company")
    verdict = interrupt({"request_id": req_id})
    return {"request_id": req_id, "verdict": verdict}


builder = StateGraph(PurchaseState)
builder.add_node("request_approval", request_approval)
builder.add_edge(START, "request_approval")
builder.add_edge("request_approval", END)
agent = builder.compile(checkpointer=checkpointer)

agent.invoke({"item": "Emergency server upgrade", "amount": 9000.0,
             "request_id": "", "verdict": ""}, cfg)

req_id = queue.pending()[0]["request_id"]
print(f"request {req_id} assigned to l1-reviewer@company")
print(f"overdue right now? {is_overdue(queue.get(req_id))}")

print(f"\nWaiting {SLA_SECONDS + 0.5}s to cross the SLA...")
time.sleep(SLA_SECONDS + 0.5)
print(f"overdue now?       {is_overdue(queue.get(req_id))}")


### 3. Escalation: reassign, don't duplicate

The tempting-but-wrong implementation is "create a new request for the
senior reviewer." Don't. That leaves **two** open requests for the same piece
of work, and if both get answered - one approve, one reject - you have a race
condition with no defined outcome. The correct move is to **reassign the
same `request_id`**, so there is only ever one live decision to make, with a
full audit trail of who had it and when.

### The escalation policy: a chain of reviewers


In [ ]:
ESCALATION_CHAIN = ["l1-reviewer@company", "l2-manager@company", "vp-eng@company"]


def escalate_if_overdue(q: ApprovalQueue, request_id: str) -> bool:
    """Move an overdue request to the next reviewer in the chain.
    Returns True if it escalated, False if it was not overdue or the
    chain is exhausted."""
    req = q.get(request_id)
    if not is_overdue(req):
        return False

    try:
        current_idx = ESCALATION_CHAIN.index(req["assigned_to"])
    except ValueError:
        current_idx = -1     # assignee not in the chain (e.g. already top)

    if current_idx + 1 >= len(ESCALATION_CHAIN):
        print(f"  [{request_id}] already at the top of the chain "
              f"({req['assigned_to']}) -- no further escalation possible")
        return False

    next_reviewer = ESCALATION_CHAIN[current_idx + 1]
    q.escalate(request_id, next_reviewer, note=f"SLA of {SLA_SECONDS}s exceeded")
    print(f"  [{request_id}] escalated: {req['assigned_to']} -> {next_reviewer} "
          f"(level {req['escalation_level'] + 1})")
    return True


print("Escalation chain:", " -> ".join(ESCALATION_CHAIN))
print()
escalate_if_overdue(queue, req_id)

print(f"\ncurrent assignee: {queue.get(req_id)['assigned_to']}")
print(f"escalation log   : {queue.get(req_id)['escalation_log']}")


### Watch the chain exhaust correctly

Escalating past the top of the chain should not error, loop, or silently
invent a reviewer - it should say plainly that there is nowhere left to go,
which is itself a signal a real system needs to act on (page someone, or
auto-reject with a clear reason).

### Escalating all the way to the top, and past it


In [ ]:
print("Escalating repeatedly (simulating the SLA elapsing again each time):\n")
for i in range(3):
    time.sleep(SLA_SECONDS + 0.5)
    escalated = escalate_if_overdue(queue, req_id)
    print(f"  attempt {i + 1}: escalated={escalated}, "
          f"now assigned to {queue.get(req_id)['assigned_to']}\n")


### 4. Composing it: a single poll loop

Put the three pieces together - check for overdue requests, escalate them,
and separately check for decisions to resume - into the shape a real
scheduled job (a cron entry, a background worker) would run every few
minutes.

### The full poll cycle


In [ ]:
def poll_cycle(graph, q: ApprovalQueue) -> dict:
    """One tick of what a production poller does. Three independent jobs:
    escalate overdue requests, resume threads whose requests were decided,
    and report anything still genuinely waiting."""
    report = {"escalated": [], "resumed": [], "still_waiting": []}

    for req in q.pending():
        if escalate_if_overdue(q, req["request_id"]):
            report["escalated"].append(req["request_id"])

    for req in list(q._read().values()):
        if req["status"] in ("approved", "rejected"):
            snap = graph.get_state({"configurable": {"thread_id": req["thread_id"]}})
            if snap.next:      # still suspended -- has not been resumed yet
                graph.invoke(Command(resume=req["status"]),
                            {"configurable": {"thread_id": req["thread_id"]}})
                report["resumed"].append(req["request_id"])

    report["still_waiting"] = [r["request_id"] for r in q.pending()]
    return report


# Let the current top-of-chain reviewer decide, THEN poll.
queue.decide(req_id, approved=True, reviewer=queue.get(req_id)["assigned_to"],
            reason="Approved after escalation to VP.")

result = poll_cycle(agent, queue)
print(f"poll result: {result}")

final_snap = agent.get_state(cfg)
print(f"\nthread status: next={final_snap.next}  values={final_snap.values}")


### Key takeaways

- A suspended thread has **no built-in timeout** - the checkpointer is happy
  to wait forever. Timeout is a policy your poller computes, from the
  request's timestamps against an SLA you define.
- **Escalate by reassigning the same `request_id`**, never by creating a
  second one for the same work - a duplicate risks two conflicting decisions
  racing each other with no defined winner.
- An escalation chain should **fail visibly** when exhausted, not loop or
  invent a reviewer - that exhaustion is itself a signal something needs a
  human (or a policy) to intervene.
- A real poller does three independent jobs each tick: escalate overdue
  requests, resume threads whose requests were just decided, and report
  what's still genuinely waiting - all three shown composed above in
  `poll_cycle()`.

**Next:** `03_multi_reviewer_signoff.ipynb` - when one approval isn't enough:
N-of-M sign-off, where several reviewers must agree before the thread
resumes.